In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm

from loka.data import load_breakthrough_states
from loka.models import BreakthroughCNN
from loka.geometrical_methods import herding_order, k_center_order

import os
import pandas as pd

import torch

from cuml.manifold import TSNE

import numpy as np

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
X_flat,Y = load_breakthrough_states("/home/axelbjarkar/RU/T-404-LOKA-code/data/e.txt", two_channel=False)
X_2d,_ = load_breakthrough_states("/home/axelbjarkar/RU/T-404-LOKA-code/data/e.txt", two_channel=True)

N = len(X_2d)

In [ ]:
f"{N:,}"

In [ ]:
def tsne_fig(X,Y, perplexity=30, random_seed=None, title=None):
    emb = TSNE(n_components=2, perplexity=perplexity, random_state=random_seed).fit_transform(X)  # numpy in, numpy out
    fig, ax = plt.subplots(figsize=(8, 8))
    for label, color in [(1, "tab:green"), (-1, "tab:red")]:
        m = Y == label
        ax.scatter(emb[m, 0], emb[m, 1], s=1, alpha=0.3, c=color, label=f"Y = {label}", rasterized=True)
    ax.legend(markerscale=10)

    if title is None:
        ax.set_title(f"t-SNE of {len(X):,} Breakthrough states")
    else:
        ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    return fig

In [ ]:
linear_tsne = tsne_fig(X_flat,Y, random_seed=SEED, title=f"t-SNE of {N:,} Breakthrough states\n(Only black and white positions)");

# Piece count states

In [ ]:
def counts(X):
    # (N, 2): white count, black count per board
    return X.reshape(len(X), 2, -1).sum(axis=2).astype(np.uint8)

In [ ]:
X_flat = X_2d.reshape(len(X_2d), -1)                             # (N, 50)
X_piece_count = np.concatenate([X_flat, counts(X_2d)], axis=1)      # (N, 52)

In [ ]:
piece_count_tsne = tsne_fig(
    X_piece_count,
    Y, 
    random_seed=SEED, 
    title=f"t-SNE of {N:,} Breakthrough states\n(Black and White positions and piece counts)"
);

In [ ]:
def near_goal(X, white_moves_up=True):
    """(N, 2) booleans: [white, black] has a piece on its goal row or the row before it."""
    B = X.reshape(len(X), 2, 5, 5).astype(bool)

    top = B[:, :, :2, :].any(axis=(2, 3))     # (N, 2): any piece in rows 0-1
    bottom = B[:, :, 3:, :].any(axis=(2, 3))  # (N, 2): any piece in rows 3-4

    if white_moves_up:  # white heads for row 0, black for row 4
        return np.stack([top[:, 0], bottom[:, 1]], axis=1)
    return np.stack([bottom[:, 0], top[:, 1]], axis=1)




In [ ]:
X_near_goal = np.concatenate([X_flat, near_goal(X_2d)], axis=1)  # (N, 52)

X_near_goal

In [ ]:
near_goal_tsne = tsne_fig(
    X_near_goal,
    Y, 
    random_seed=SEED, 
    title=f"t-SNE of {N:,} Breakthrough states\n(Black and White positions and if either is close to goal)"
);

In [ ]:
X_near_goal_and_piece_count = np.concatenate([X_flat, counts(X_2d), near_goal(X_2d)], axis=1)

In [ ]:
near_goal_and_piece_count_tsne = tsne_fig(
    X_near_goal_and_piece_count,
    Y,
    random_seed=SEED,
    title=f"t-SNE of {N:,} Breakthrough states\n(Piece positions, piece counts and near-goal flags)"
);